# Phase 2 - B5 Embedding-Similarity Replay (Kaggle)

This notebook runs B5 on the frozen five-language XNLI protocol using the shared Phase 1 candidate-label prompt and LoRA trainer. A 125-example uniform reservoir stores examples from completed languages. Before each later language stage, the pretrained Llama backbone (with LoRA adapters disabled) creates attention-masked mean-pooled prompt embeddings; each current example replays its nearest prior example by cosine similarity. Replay is 1:1, and current examples enter the reservoir after their stage. This is a documented nearest-neighbor operationalization of embedding-similarity replay, not an exact reproduction of a specific paper implementation.

All five orders are selected by default. To fit multiple Kaggle sessions, change `ORDER_IDS` below to run one or two orders, save a notebook version with output, then attach that saved output as input to the next session and set `PREVIOUS_ARTIFACTS_DIR` to its `phase2_b5_artifacts` directory. Completed orders are validated and skipped; an interrupted order restarts from that order's beginning.

## Before running
1. Enable a Kaggle GPU and Internet.
2. Push the B5 runner changes to the public repository, since this notebook clones the repository before execution.
3. Accept the gated model terms for `meta-llama/Llama-3.2-1B` and add a Kaggle secret named `HF_TOKEN`.
4. Optional Drive backup: enable the Google Drive API, create a Google OAuth **Desktop app** client, and locally run `python -m pip install google-auth-oauthlib` then `python phase2\\create_drive_oauth_token.py <client-secrets.json>`. Add the client ID, client secret, and printed refresh token as Kaggle Secrets named `GOOGLE_OAUTH_CLIENT_ID`, `GOOGLE_OAUTH_CLIENT_SECRET`, and `GOOGLE_OAUTH_REFRESH_TOKEN`; then set `DRIVE_BACKUP_ENABLED = True` below. Never paste credentials into notebook code, GitHub, or chat. Google notes that refresh tokens for external OAuth apps left in Testing expire after 7 days.
5. Run all cells, then use **Save Version** with notebook output enabled to preserve `/kaggle/working/phase2_b5_artifacts`; Drive upload is an additional copy, not a replacement.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Mani-S-49/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if REPO_ROOT.exists():
    raise FileExistsError(
        f"Repository directory already exists: {REPO_ROOT}. Restart the Kaggle session or choose a fresh path."
    )
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
repo_revision = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
print("Cloned repository revision:", repo_revision)

requirements_file = REPO_ROOT / "phase2" / "requirements-phase2.txt"
if not requirements_file.is_file():
    raise FileNotFoundError(f"Requirements file not found: {requirements_file}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)],
    check=True,
)
sys.path.insert(0, str(REPO_ROOT))

runner_path = REPO_ROOT / "phase2" / "run_baseline.py"
if not runner_path.is_file():
    raise FileNotFoundError(f"Missing B5 runner: {runner_path}")
help_result = subprocess.run(
    [sys.executable, "-m", "phase2.run_baseline", "--help"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(help_result.stdout)
if help_result.returncode != 0 or "B5" not in help_result.stdout:
    raise RuntimeError(
        "The cloned repository does not expose B5. Push the B5 runner changes to the default branch, then restart this Kaggle session."
    )

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token

PROTOCOL_PATH = REPO_ROOT / "phase2" / "configs" / "pilot_protocol.json"
ARTIFACT_ROOT = Path("/kaggle/working/phase2_b5_artifacts")
PREVIOUS_ARTIFACTS_DIR = None  # Set to the mounted prior output's phase2_b5_artifacts directory to continue.
DRIVE_BACKUP_ENABLED = False  # Enable only after adding the three Google OAuth values to Kaggle Secrets.
print({
    "repo": str(REPO_ROOT),
    "revision": repo_revision,
    "artifact_root": str(ARTIFACT_ROOT),
    "runner_file": str(runner_path),
})


In [ ]:
from phase2.preflight import collect_preflight
from phase2.prepare_data import prepare_xnli
from phase2.protocol import load_protocol
from phase2.run_baseline import validate_prepared_artifacts
import json

preflight = collect_preflight(require_cuda=True, check_gated_model=True)
print(json.dumps(preflight, indent=2, sort_keys=True))
protocol = load_protocol(PROTOCOL_PATH)

if not (ARTIFACT_ROOT / "protocol_manifest.json").is_file():
    if ARTIFACT_ROOT.exists():
        raise FileExistsError(
            "A partial working artifact directory exists without a protocol manifest. Inspect it before continuing."
        )
    if PREVIOUS_ARTIFACTS_DIR is not None:
        previous = Path(PREVIOUS_ARTIFACTS_DIR)
        if not (previous / "protocol_manifest.json").is_file():
            raise FileNotFoundError(f"Previous artifact manifest not found under {previous}")
        shutil.copytree(previous, ARTIFACT_ROOT)
        print("Copied previous Kaggle B5 artifacts from:", previous)
    else:
        prepare_xnli(PROTOCOL_PATH, ARTIFACT_ROOT)
        print("Prepared the fixed XNLI subsets.")
prepared_manifest = validate_prepared_artifacts(protocol, ARTIFACT_ROOT)

ALL_ORDER_IDS = [order["id"] for order in protocol["orders"]]
ORDER_IDS = ALL_ORDER_IDS.copy()  # For separate sessions, select one or two order IDs here.
if len(ALL_ORDER_IDS) != 5:
    raise ValueError(f"B5 expects five protocol orders; found {len(ALL_ORDER_IDS)}.")
if not ORDER_IDS or len(set(ORDER_IDS)) != len(ORDER_IDS) or set(ORDER_IDS) - set(ALL_ORDER_IDS):
    raise ValueError(f"Choose unique protocol order IDs from {sorted(ALL_ORDER_IDS)}.")
print({
    "languages": protocol["dataset"]["languages"],
    "orders_to_run": ORDER_IDS,
    "buffer_capacity": protocol["replay"]["buffer_capacity_examples"],
    "data_manifest_sha256": prepared_manifest["data_manifest_sha256"],
})


In [ ]:
command = [
    sys.executable,
    "-m",
    "phase2.run_baseline",
    "--method",
    "B5",
    "--protocol",
    str(PROTOCOL_PATH),
    "--artifact-root",
    str(ARTIFACT_ROOT),
]
for order_id in ORDER_IDS:
    command.extend(["--order-id", order_id])
print("RUN COMMAND:")
print(command)
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
from phase2.protocol import build_manifest
from phase2.results import validate_result_record
import json
import statistics

run_manifest = build_manifest(protocol)
run_manifest["data_artifacts"] = prepared_manifest["data_artifacts"]
run_manifest["data_manifest_sha256"] = prepared_manifest["data_manifest_sha256"]
expected_runs = {
    run["run_id"]: run
    for run in run_manifest["runs"]
    if run["method_id"] == "B5"
}
if len(expected_runs) != 5:
    raise ValueError(f"Expected five B5 runs in the current manifest; found {len(expected_runs)}.")
completed = {}
for run_id, run in expected_runs.items():
    result_path = ARTIFACT_ROOT / "results" / f"{run_id}.json"
    if result_path.is_file():
        record = json.loads(result_path.read_text(encoding="utf-8"))
        validate_result_record(record, run_manifest)
        completed[run["order_id"]] = record

metric_names = ("ACC", "BWT", "FWT", "F")
summary = {
    "method_id": "B5",
    "method_interpretation": "Frozen-base nearest-neighbor embedding-similarity replay adaptation",
    "protocol_id": run_manifest["protocol_id"],
    "protocol_sha256": run_manifest["protocol_sha256"],
    "data_manifest_sha256": run_manifest["data_manifest_sha256"],
    "completed_orders": len(completed),
    "expected_orders": len(expected_runs),
    "complete": len(completed) == len(expected_runs),
    "per_order": {
        order_id: {name: record["metrics"][name] for name in metric_names}
        for order_id, record in sorted(completed.items())
    },
}
if completed:
    summary["across_order_summary"] = {
        name: {
            "mean": statistics.mean(record["metrics"][name] for record in completed.values()),
            "population_std": statistics.pstdev(record["metrics"][name] for record in completed.values()),
        }
        for name in metric_names
    }
summary_path = ARTIFACT_ROOT / "B5_summary.json"
temporary_path = summary_path.with_suffix(".json.tmp")
temporary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
temporary_path.replace(summary_path)
print(json.dumps(summary, indent=2, sort_keys=True))
print(f"Validated {len(completed)}/{len(expected_runs)} B5 orders. Save the Kaggle notebook version/output to preserve artifacts:", ARTIFACT_ROOT)


## Saved outputs and continuing in another Kaggle session

- `/kaggle/working/phase2_b5_artifacts/protocol_manifest.json` and `data/`: fixed selected examples and checksums.
- `phase2_b5_artifacts/checkpoints/<run_id>/`: adapters and tokenizers after each language stage.
- `phase2_b5_artifacts/ledgers/<run_id>/`: evaluation matrix and ledger.
- `phase2_b5_artifacts/results/<run_id>.json`: validated result for each completed order, with replay composition, cosine similarities, and timing.
- `phase2_b5_artifacts/B5_summary.json`: validated results and summary; `complete` is true only after all five orders.

To continue in a new Kaggle session, attach the previous notebook output as an input, set `PREVIOUS_ARTIFACTS_DIR` to its mounted `phase2_b5_artifacts` directory, choose the remaining `ORDER_IDS`, then save the new output again. Keep the same data manifest and code revision for every order. The 125-example memory is sampled uniformly from previously seen languages; each later stage uses the nearest stored example for every current prompt by cosine similarity, then adds the current stage to memory. Embeddings use the pretrained Llama backbone with LoRA adapters disabled. Never tune using test scores.


In [ ]:
import json
import os
import shutil
from IPython.display import FileLink, display

summary_path = ARTIFACT_ROOT / "B5_summary.json"
if not summary_path.is_file():
    raise FileNotFoundError(f"B5 summary not found: {summary_path}")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
if (
    summary.get("method_id") != "B5"
    or summary.get("expected_orders") != 5
    or not isinstance(summary.get("completed_orders"), int)
    or not 0 <= summary["completed_orders"] <= 5
    or summary.get("complete") is not (summary["completed_orders"] == 5)
):
    raise RuntimeError(
        "Not creating the B5 ZIP: the summary has invalid completion metadata."
    )

backup_status = "complete" if summary["complete"] else "partial"
os.chdir("/kaggle/working")
archive_path = Path(
    shutil.make_archive(
        f"phase2_b5_artifacts_{backup_status}",
        "zip",
        root_dir=ARTIFACT_ROOT.parent,
        base_dir=ARTIFACT_ROOT.name,
    )
)
if not archive_path.is_file() or archive_path.stat().st_size == 0:
    raise RuntimeError(f"B5 ZIP creation failed: {archive_path}")
print(f"Created {backup_status} B5 archive: {archive_path} ({archive_path.stat().st_size / (1024 ** 2):.1f} MiB)")
print("Click the link below to download the B5 output ZIP:")
display(FileLink(archive_path.name, result_html_prefix="Download: "))


In [ ]:
if DRIVE_BACKUP_ENABLED:
    from datetime import datetime, timezone
    import requests

    drive_client_id = secrets.get_secret("GOOGLE_OAUTH_CLIENT_ID")
    drive_client_secret = secrets.get_secret("GOOGLE_OAUTH_CLIENT_SECRET")
    drive_refresh_token = secrets.get_secret("GOOGLE_OAUTH_REFRESH_TOKEN")
    if not all((drive_client_id, drive_client_secret, drive_refresh_token)):
        raise RuntimeError("Google Drive OAuth Kaggle Secrets must all be non-empty.")

    try:
        token_response = requests.post(
            "https://oauth2.googleapis.com/token",
            data={
                "client_id": drive_client_id,
                "client_secret": drive_client_secret,
                "refresh_token": drive_refresh_token,
                "grant_type": "refresh_token",
            },
            timeout=30,
        )
    except requests.RequestException:
        raise RuntimeError("Could not contact Google's OAuth endpoint.") from None
    if token_response.status_code != 200:
        raise RuntimeError(
            f"Google OAuth token refresh failed (HTTP {token_response.status_code}); check the Kaggle Secrets and OAuth consent."
        )
    access_token = token_response.json().get("access_token")
    if not access_token:
        raise RuntimeError("Google OAuth response did not contain an access token.")

    api_root = "https://www.googleapis.com/drive/v3"
    api_headers = {"Authorization": f"Bearer {access_token}"}

    def drive_json_request(method, url, *, params=None, payload=None, headers=None):
        request_headers = dict(api_headers)
        if headers:
            request_headers.update(headers)
        try:
            response = requests.request(
                method,
                url,
                params=params,
                json=payload,
                headers=request_headers,
                timeout=60,
            )
        except requests.RequestException:
            raise RuntimeError("Google Drive API request failed; check the Kaggle Internet setting.") from None
        if response.status_code not in (200, 201):
            raise RuntimeError(
                f"Google Drive API request failed (HTTP {response.status_code}); verify OAuth scope and Drive quota."
            )
        return response.json()

    folder_name = "Kaggle_Phase2_B5_Backups"
    folder_query = (
        f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
    )
    folder_listing = drive_json_request(
        "GET",
        f"{api_root}/files",
        params={"q": folder_query, "spaces": "drive", "pageSize": 10, "fields": "files(id,name)"},
    )
    folders = folder_listing.get("files", [])
    if folders:
        drive_folder_id = folders[0]["id"]
    else:
        new_folder = drive_json_request(
            "POST",
            f"{api_root}/files",
            payload={"name": folder_name, "mimeType": "application/vnd.google-apps.folder"},
            params={"fields": "id,name"},
        )
        drive_folder_id = new_folder["id"]

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    drive_archive_name = f"phase2_b5_artifacts_{backup_status}_{timestamp}.zip"
    archive_size = archive_path.stat().st_size
    try:
        upload_init = requests.post(
            f"https://www.googleapis.com/upload/drive/v3/files",
            params={"uploadType": "resumable", "fields": "id,name,size"},
            json={"name": drive_archive_name, "parents": [drive_folder_id]},
            headers={
                **api_headers,
                "X-Upload-Content-Type": "application/zip",
                "X-Upload-Content-Length": str(archive_size),
            },
            timeout=60,
        )
    except requests.RequestException:
        raise RuntimeError("Could not start the Google Drive archive upload.") from None
    if upload_init.status_code != 200 or not upload_init.headers.get("Location"):
        raise RuntimeError(
            f"Google Drive did not start the archive upload (HTTP {upload_init.status_code})."
        )
    upload_url = upload_init.headers["Location"]
    chunk_size = 8 * 1024 * 1024
    uploaded = 0
    upload_result = None
    with archive_path.open("rb") as archive_file:
        while uploaded < archive_size:
            chunk = archive_file.read(min(chunk_size, archive_size - uploaded))
            if not chunk:
                raise RuntimeError("B5 ZIP ended before the expected upload size.")
            chunk_end = uploaded + len(chunk) - 1
            try:
                chunk_response = requests.put(
                    upload_url,
                    data=chunk,
                    headers={
                        "Content-Length": str(len(chunk)),
                        "Content-Range": f"bytes {uploaded}-{chunk_end}/{archive_size}",
                    },
                    timeout=(30, 300),
                )
            except requests.RequestException:
                raise RuntimeError("Google Drive archive upload was interrupted; rerun the backup cell.") from None
            uploaded += len(chunk)
            if chunk_response.status_code in (200, 201):
                upload_result = chunk_response.json()
                break
            if chunk_response.status_code != 308:
                raise RuntimeError(
                    f"Google Drive archive upload failed (HTTP {chunk_response.status_code})."
                )
    if not upload_result or not upload_result.get("id"):
        raise RuntimeError("Google Drive did not confirm the completed B5 archive upload.")
    print(f"Uploaded {drive_archive_name} to Google Drive ({archive_size / (1024 ** 2):.1f} MiB).")
    del access_token, drive_client_id, drive_client_secret, drive_refresh_token, api_headers, upload_url
else:
    print("Google Drive backup is disabled. The ZIP is available for Kaggle download above.")
